# 07b · From NanoGPT to a Modern Decoder
**Time:** 60 minutes · **Prerequisite:** 07 · **Downloads:** none

Build on the classic GPT lesson with a locally trained byte-level BPE tokenizer,
rotary position embeddings (RoPE), RMSNorm, SwiGLU, and grouped-query attention
(GQA). These are concrete components used in [MLX-LM's Llama implementation](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/models/llama.py).
They are not a recipe for achieving frontier quality on a tiny corpus.

**Objectives:** explain each change; compare parameter counts and held-out loss
at equal token budgets; verify cached decoding; measure the KV-memory saving.
Predict which change reduces cache memory before running anything.

In [ ]:
import json
import math
import time
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import numpy as np
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from modern_decoder import Decoder, cache_bytes
from mlx_nlp_utils import make_train_step, count_parameters, load_sample_corpus
from experiment_utils import write_record

mx.random.seed(0)
corpus, _, _ = load_sample_corpus()
# Split text before fitting a tokenizer. This tiny split is educational, not a benchmark.
boundary = int(len(corpus) * 0.8)
train_text, held_out_text = corpus[:boundary], corpus[boundary:]


## 1. Subwords without a pretrained tokenizer download
Byte-level BPE begins with a byte alphabet and learns frequent pair merges.
Unlike a character vocabulary built from all text, this tokenizer is fitted
on the training partition only. The byte alphabet handles unseen Unicode.
A larger vocabulary reduces sequence length but increases embedding parameters.
Compare token counts on *both* partitions. Perplexities from different
tokenizers are not directly comparable; our architecture ablations share one.

In [ ]:
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
tokenizer.train_from_iterator([train_text], trainers.BpeTrainer(
    vocab_size=320, special_tokens=["[UNK]"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
train_tokens = mx.array(tokenizer.encode(train_text).ids)
val_tokens = mx.array(tokenizer.encode(held_out_text).ids)
for name, text in [("train", train_text), ("held out", held_out_text)]:
    encoded = tokenizer.encode(text)
    assert tokenizer.decode(encoded.ids) == text
    print(name, "characters:", len(text), "subword tokens:", len(encoded.ids))
print(tokenizer.encode("unified memory 🧠").tokens)


## 2. Read the implementation
Open [`modern_decoder.py`](modern_decoder.py). It is deliberately small.

| Component | Classic version | Modern variant | Hypothesis |
|---|---|---|---|
| Position | Added learned vectors | Rotate Q/K with RoPE | Relative offsets enter attention scores |
| Normalization | LayerNorm | RMSNorm | Remove mean subtraction |
| MLP | GELU with 4d hidden width | SiLU gate multiplied by an up projection | Multiplicative feature selection |
| Attention | One K/V head per query head | Share K/V heads across query groups | Smaller KV cache |

The SwiGLU width is rounded near 8d/3 to keep its three matrices near the
parameter cost of the classic two-matrix MLP. GQA saves parameters too, so
this is an equal-token-budget comparison, not an exactly equal-parameter one.
RoPE is an architectural transform, not a learned position table.

🤔 Predict whether any variant reliably wins with only about 2K characters.

In [ ]:
configs = {
    "classic": dict(rope=False, rmsnorm=False, swiglu=False, kv_heads=4),
    "+ RoPE": dict(rope=True, rmsnorm=False, swiglu=False, kv_heads=4),
    "+ RMSNorm": dict(rope=True, rmsnorm=True, swiglu=False, kv_heads=4),
    "+ SwiGLU": dict(rope=True, rmsnorm=True, swiglu=True, kv_heads=4),
    "+ GQA": dict(rope=True, rmsnorm=True, swiglu=True, kv_heads=2),
}
block_size, batch_size = 32, 4
train_steps = 60

def batches(data):
    starts = mx.random.randint(0, len(data) - block_size, (batch_size, 1))
    positions = starts + mx.arange(block_size)[None]
    return data[positions], data[positions + 1]

def loss_fn(model, x, y):
    return nn.losses.cross_entropy(model(x), y, reduction="mean")

rows = []
for name, options in configs.items():
    mx.random.seed(0)
    model = Decoder(tokenizer.get_vocab_size(), **options)
    optimizer = optim.AdamW(learning_rate=3e-4)
    step, state = make_train_step(model, optimizer, loss_fn)
    # Reset sampling RNG so each configuration sees the same windows.
    mx.random.seed(1)
    start = time.perf_counter()
    for _ in range(train_steps):
        x, y = batches(train_tokens)
        loss = step(x, y)
        mx.eval(loss, state)
    seconds = time.perf_counter() - start
    model.eval()
    x, y = val_tokens[:-1][None], val_tokens[1:][None]
    val_loss = loss_fn(model, x, y).item()
    rows.append({"variant": name, "parameters": count_parameters(model),
                 "validation_nll": val_loss, "perplexity": math.exp(val_loss),
                 "training_seconds": seconds, "tokens_seen": train_steps * batch_size * block_size})
for row in rows:
    print(row)


## 3. Cache correctness comes before speed
Autoregressive decoding can retain old keys and values and process only the
new tokens. RoPE must use the **absolute offset** of each new token. The mask
for a cached multi-token suffix has T queries and prefix+T keys; a plain T×T
triangle would be wrong. Test both one-token and multi-token chunks against
an uncached forward pass before benchmarking.

In [ ]:
tokens = train_tokens[:24][None]
full = model(tokens)
cache, chunks = None, []
for start, end in [(0, 7), (7, 8), (8, 15), (15, 24)]:
    logits, cache = model(tokens[:, start:end], cache=cache, return_cache=True)
    chunks.append(logits)
cached = mx.concatenate(chunks, axis=1)
np.testing.assert_allclose(np.array(full), np.array(cached), atol=2e-5, rtol=2e-5)
print("Cache equivalence passed; live K/V bytes:", cache_bytes(cache))


## 4. Verify the memory model
For ordinary floating-point K/V tensors, bytes = 2 × layers × batch × tokens ×
KV heads × head dimension × bytes per value. GQA reduces KV heads without
reducing query heads. This helper stores concatenated tensors for clarity;
production caches allocate blocks and can use quantization.

In [ ]:
memory_rows = []
for kv_heads in [4, 2, 1]:
    candidate = Decoder(tokenizer.get_vocab_size(), kv_heads=kv_heads)
    _, cache = candidate(tokens, return_cache=True)
    mx.eval(cache)
    expected = 2 * 2 * 1 * tokens.shape[1] * kv_heads * 16 * 4
    assert cache_bytes(cache) == expected
    memory_rows.append({"kv_heads": kv_heads, "cache_bytes": cache_bytes(cache)})
print(memory_rows)
print(write_record("07b-decoder", settings={"configs": configs, "steps": train_steps,
    "tokenizer": json.loads(tokenizer.to_str()), "split_character": boundary,
    "batch_seed": 1}, metrics={"training": rows, "cache": memory_rows},
    data_files=[Path('../data/text_gen_samples/corpus.txt')], seed=0))


## Exercises and limits
1. Repeat with three initialization seeds. Does the ranking survive?
2. Increase vocabulary size and compare validation **bits per byte**, accounting
   for the held-out byte count. Why is perplexity alone misleading across tokenizers?
3. Break the RoPE offset and observe the cache-equivalence assertion fail.
4. Hold parameter count approximately fixed while varying KV heads.

No architecture win is claimed from this tiny run. Extend the document-level
split to a real corpus before drawing quality conclusions. Next, notebook 11
measures inference with a pinned pretrained model.